In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieLensLSH") \
    .getOrCreate()

sc = spark.sparkContext

26/03/02 21:40:37 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/02 21:40:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 21:40:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 21:40:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [14]:
from pyspark import SparkContext
import random

In [2]:
file_path = "/home/rajesh/CSL7100/Assignment2/MovieLens/ml-100k/u.data"

# Read file as RDD of lines
data = sc.textFile(file_path)

In [6]:
# Split each line and extract (user, movie)
user_movies = (
    data
    .map(lambda line: line.split('\t'))            # split each line
    .map(lambda x: (int(x[0]), int(x[1])))         # (user_id, movie_id)
    .groupByKey()                                  # group movies by user
    .mapValues(lambda movies: set(movies))         # convert to set (remove duplicates)
    .cache()                                       # cache for reuse
)

users = user_movies.keys().collect()               # list of user IDs

In [7]:
def true_jaccard(set1, set2):
    return float(len(set1 & set2)) / len(set1 | set2)   # |A∩B| / |A∪B|

In [8]:
# Create all user pairs using cartesian
user_pairs = (
    user_movies.cartesian(user_movies)
    .filter(lambda x: x[0][0] < x[1][0])   # avoid duplicate pairs
)

# Compute exact similarity
exact_pairs = (
    user_pairs
    .map(lambda x: (
        (x[0][0], x[1][0]),                     # (user1, user2)
        true_jaccard(x[0][1], x[1][1])          # similarity
    ))
    .filter(lambda x: x[1] >= 0.5)              # keep only >= 0.5
    .map(lambda x: x[0])                        # keep only user pair
)

exact_set = set(exact_pairs.collect())          # collect for comparison
print("Exact pairs >= 0.5:", len(exact_set))

[Stage 3:=============================>                             (2 + 2) / 4]

Exact pairs >= 0.5: 10


In [9]:
def generate_hash_functions(num_hashes):

    p = 2003    # prime > 1682 (number of movies)

    hash_funcs = []
    for _ in range(num_hashes):
        a = random.randint(1, 1682)    # random coefficient
        b = random.randint(0, 1682)
        hash_funcs.append((a, b, p))

    return hash_funcs

In [10]:
def compute_signature(movie_set, hash_funcs):

    signature = []

    for (a, b, p) in hash_funcs:

        # Compute hash for every movie in user's set
        min_hash = min((a * m + b) % p for m in movie_set)

        signature.append(min_hash)   # store minimum hash value

    return signature

In [11]:
def estimate_jaccard(sig1, sig2):

    matches = 0

    for i in range(len(sig1)):
        if sig1[i] == sig2[i]:
            matches += 1

    return matches / float(len(sig1))   # fraction of matching positions

In [12]:
def run_experiment(num_hashes, runs=5):

    false_pos_total = 0
    false_neg_total = 0

    for r in range(runs):

        print("Run", r+1, "with", num_hashes, "hashes")

        hash_funcs = generate_hash_functions(num_hashes)

        bc_hash = sc.broadcast(hash_funcs)   # send hash funcs to all workers

        # Compute signature for each user
        signatures = (
            user_movies
            .mapValues(lambda movies: compute_signature(movies, bc_hash.value))
            .cache()
        )

        # Compare all pairs using estimated similarity
        approx_pairs = (
            signatures.cartesian(signatures)
            .filter(lambda x: x[0][0] < x[1][0])     # avoid duplicates
            .map(lambda x: (
                (x[0][0], x[1][0]),
                estimate_jaccard(x[0][1], x[1][1])
            ))
            .filter(lambda x: x[1] >= 0.5)
            .map(lambda x: x[0])
        )

        approx_set = set(approx_pairs.collect())

        # Compute errors
        false_pos = len(approx_set - exact_set)
        false_neg = len(exact_set - approx_set)

        false_pos_total += false_pos
        false_neg_total += false_neg

    print("\nResults for", num_hashes, "hashes")
    print("Average False Positives:", false_pos_total / runs)
    print("Average False Negatives:", false_neg_total / runs)
    print("------------------------------------------------")

In [15]:
run_experiment(50)


Run 1 with 50 hashes


26/03/02 21:46:05 WARN BlockManager: Block rdd_10_0 already exists on this machine; not re-adding it
26/03/02 21:46:05 WARN BlockManager: Block rdd_10_1 already exists on this machine; not re-adding it


Run 2 with 50 hashes
Run 3 with 50 hashes


26/03/02 21:46:06 WARN BlockManager: Block rdd_16_1 already exists on this machine; not re-adding it


Run 4 with 50 hashes


26/03/02 21:46:06 WARN BlockManager: Block rdd_19_1 already exists on this machine; not re-adding it
26/03/02 21:46:06 WARN BlockManager: Block rdd_19_0 already exists on this machine; not re-adding it
                                                                                

Run 5 with 50 hashes


26/03/02 21:46:07 WARN BlockManager: Block rdd_22_0 already exists on this machine; not re-adding it
26/03/02 21:46:07 WARN BlockManager: Block rdd_22_1 already exists on this machine; not re-adding it



Results for 50 hashes
Average False Positives: 160.8
Average False Negatives: 2.0
------------------------------------------------


In [16]:
run_experiment(100)

Run 1 with 100 hashes


26/03/02 21:46:33 WARN BlockManager: Block rdd_25_1 already exists on this machine; not re-adding it
                                                                                

Run 2 with 100 hashes


26/03/02 21:46:34 WARN BlockManager: Block rdd_28_0 already exists on this machine; not re-adding it
26/03/02 21:46:34 WARN BlockManager: Block rdd_28_1 already exists on this machine; not re-adding it
                                                                                

Run 3 with 100 hashes


...

Run 4 with 100 hashes


26/03/02 21:46:36 WARN BlockManager: Block rdd_34_0 already exists on this machine; not re-adding it
                                                                                

Run 5 with 100 hashes


26/03/02 21:46:37 WARN BlockManager: Block rdd_37_0 already exists on this machine; not re-adding it
26/03/02 21:46:37 WARN BlockManager: Block rdd_37_1 already exists on this machine; not re-adding it
[Stage 23:>                                                         (0 + 4) / 4]


Results for 100 hashes
Average False Positives: 40.0
Average False Negatives: 3.4
------------------------------------------------


In [17]:
run_experiment(200)

Run 1 with 200 hashes


26/03/02 21:46:57 WARN BlockManager: Block rdd_40_0 already exists on this machine; not re-adding it
26/03/02 21:46:57 WARN BlockManager: Block rdd_40_1 already exists on this machine; not re-adding it
                                                                                

Run 2 with 200 hashes


26/03/02 21:46:59 WARN BlockManager: Block rdd_43_0 already exists on this machine; not re-adding it
                                                                                

Run 3 with 200 hashes


...

Run 4 with 200 hashes


26/03/02 21:47:02 WARN BlockManager: Block rdd_49_1 already exists on this machine; not re-adding it
                                                                                

Run 5 with 200 hashes


26/03/02 21:47:03 WARN BlockManager: Block rdd_52_0 already exists on this machine; not re-adding it
26/03/02 21:47:03 WARN BlockManager: Block rdd_52_1 already exists on this machine; not re-adding it



Results for 200 hashes
Average False Positives: 6.8
Average False Negatives: 1.6
------------------------------------------------


In [58]:
if spark:
    spark.stop()